# Pipeline de Treinamento — MultiHead

Executa o pipeline completo:
1. Geração de dataset a partir do perfil + dados diários
2. Normalização e split temporal (treino / validação / teste)
3. Treinamento com PyTorch Lightning
4. Avaliação com métricas denormalizadas
5. Exportação ONNX
6. Relatório PDF

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

In [ ]:
from api.config.dataset_config import DatasetConfig
from api.config.training_config import TrainingConfig
from api.config.model_config import ModelConfig
from api.config.output_config import OutputConfig

from moviasai.data.utils import load_raw_data
from moviasai.forecasting.training_pipeline import MultiHeadTrainingPipeline

## Configurações

In [ ]:
def run_training(target: str) -> dict:
    """Executa o pipeline completo de treinamento para o target especificado ('km' ou 'h')."""
    dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')
    training_cfg = TrainingConfig.from_yaml('../config/training_config.yaml')
    model_cfg = ModelConfig.from_yaml('../config/model_config.yaml')
    output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')

    df_daily = load_raw_data(output_cfg.train_data_path(target), target=target)

    print(f'Target:     {target}')
    print(f'Epochs:     {training_cfg.trainer.max_epochs}')
    print(f'Batch size: {training_cfg.data.batch_size}')
    print(f'ONNX:       {training_cfg.onnx.enabled}')

    pipeline = MultiHeadTrainingPipeline.from_config(
        df_daily=df_daily.to_pandas(),
        dataset_cfg=dataset_cfg,
        training_cfg=training_cfg,
        model_cfg=model_cfg,
        output_config=output_cfg,
        target=target,
    )

    metrics = pipeline.run()
    for split_name in ('val', 'test'):
        if split_name not in metrics:
            continue
        maint = metrics[split_name]['maintenance']
        print(f'\n=== {split_name.upper()} ===')
        for k_label in ('k2', 'k3', 'k4'):
            m = maint[k_label]
            k = int(k_label[1])
            print(f"\n  Limite {k} semanas:")
            print(f"    Erro médio:          {m['mean_error']:+.2f} dias")
            print(f"    Erro absoluto médio: {m['mae_days']:.2f} dias")
            print(f"    P90 erro:            {m['p90_error']:+.2f} dias")
            print(f"    % atrasos:           {m['pct_late']:.1f}%")
            print(f"    % adiantamentos:     {m['pct_early']:.1f}%")
    return metrics

## Pipeline completo

Executa todas as etapas sequencialmente e gera o relatório PDF.

In [4]:
metrics_km = run_training('km')
metrics_h = run_training('h')

Target:     km
Epochs:     10
Batch size: 64
ONNX:       True
✓ Modelo PKL carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.pkl
  Nome: stage2_km
  Features: 5
  Classes: 3

✓ Carregado: 6718 veículos, 229760 versões
  - Extractors: 4
    • segmentation_features_km: 5 features
    • weekday_features_km: 49 features
    • month_phase_features_km: 15 features
    • monthly_cycle_features_km: 7 features
  - Classificador: stage2_km
✓ Dataset carregado: C:\Users\f0pi\git\apimovias\data\.dataset_cache\km
  • Amostras: 115,803
  • Features: 78
  • X_recent: (115803, 28)
  • Heads: 4


Seed set to 13


train_test_split temporal:
  Cutoff:       2025-08-17
  Treino:       100,600 amostras (86.9%) | 2025-03-23 → 2025-08-17 (22 semanas)
  Teste:        15,203 amostras (13.1%) | 2025-08-24 → 2025-09-07 (3 semanas)
train_test_split temporal:
  Cutoff:       2025-07-20
  Treino:       80,695 amostras (80.2%) | 2025-03-23 → 2025-07-20 (18 semanas)
  Teste:        19,905 amostras (19.8%) | 2025-07-27 → 2025-08-17 (4 semanas)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type       | Params | Mode 
--------------------------------------------------------
0 | encoder_general  | Sequential | 6.1 K  | train
1 | encoder_temporal | Sequential | 3.9 K  | train
2 | fusion           | Sequential | 6.2 K  | train
3 | head_agg         | Linear     | 260    | train
4 | head_daily       | Linear     | 455    | train
5 | loss_heads_fn    | HuberLoss  | 0      | train
6 | loss_daily_fn    | L1Loss     | 0      | train
--------------------------------------------------------
16.9 K    Trainable params
0         Non-trainable params
16.9 K    Total params
0.068     Total estimated model params size (MB)
21        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.



  MÉTRICAS — VAL

  HEADS (agregado semanal):
    head_0 (7d):  MAE=7.76  RMSE=11.53
    head_1 (7d):  MAE=8.87  RMSE=13.27
    head_2 (7d):  MAE=9.74  RMSE=14.46
    head_3 (7d):  MAE=10.38  RMSE=15.34

  DAILY (dias 1–7):
    MAE médio:  1.51
    RMSE médio: 2.58

  DECISÃO DE MANUTENÇÃO:

  Limite: 2 semanas (k=2)
    Erro médio:          +1.54 dias
    Erro absoluto médio: 1.92 dias
    P90 erro:            +7.14 dias
    % atrasos:           30.5%
    % adiantamentos:     5.2%

  Limite: 3 semanas (k=3)
    Erro médio:          +0.13 dias
    Erro absoluto médio: 0.19 dias
    P90 erro:            +0.00 dias
    % atrasos:           5.0%
    % adiantamentos:     0.5%

  Limite: 4 semanas (k=4)
    Erro médio:          -0.01 dias
    Erro absoluto médio: 0.01 dias
    P90 erro:            +0.00 dias
    % atrasos:           0.2%
    % adiantamentos:     0.2%


  MÉTRICAS — TEST

  HEADS (agregado semanal):
    head_0 (7d):  MAE=7.66  RMSE=11.82
    head_1 (7d):  MAE=8.91  RMSE=13.

W0423 18:20:38.067000 54344 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0423 18:20:38.069000 54344 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0423 18:20:38.071000 54344 site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `VehicleForecastingModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VehicleForecastingModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Relatório gerado: C:\Users\f0pi\git\apimovias\logs\training\h\training_report_h.pdf

=== VAL ===

  Limite 2 semanas:
    Erro médio:          +1.54 dias
    Erro absoluto médio: 1.92 dias
    P90 erro:            +7.14 dias
    % atrasos:           30.5%
    % adiantamentos:     5.2%

  Limite 3 semanas:
    Erro médio:          +0.13 dias
    Erro absoluto médio: 0.19 dias
    P90 erro:            +0.00 dias
    % atrasos:           5.0%
    % adiantamentos:     0.5%

  Limite 4 semanas:
    Erro médio:      

## Métricas

In [5]:
dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')



## Execução passo a passo (opcional)

Para depurar ou inspecionar resultados intermediários, execute cada etapa separadamente.

In [6]:
# pipeline = TrainingPipeline.from_config(
#     df_daily=df_daily.to_pandas(),
#     dataset_cfg=dataset_cfg,
#     training_cfg=training_cfg,
#     model_cfg=model_cfg,
#     output_config=output_cfg,
#     target=TARGET,
# )

In [7]:
# # 2. Normalizar e dividir
# pipeline.normalize_and_split()
# print(f'Treino:     {len(pipeline.train_ds)}')
# print(f'Validação:  {len(pipeline.val_ds) if pipeline.val_ds else "N/A"}')
# print(f'Teste:      {len(pipeline.test_ds) if pipeline.test_ds else "N/A"}')

In [8]:
# # 3. Treinar
# pipeline.train()

In [9]:
# # 4. Avaliar
# metrics = pipeline.evaluate()

In [10]:
# # 5. Exportar ONNX
# pipeline.export_onnx()

In [11]:
# # 6. Gerar relatório PDF
# pipeline.generate_report()